# 01 – Exploratory data analysis

Runs both on Kaggle (data mounted under `/kaggle/input`) and locally (data in `DATA_DIR`).

In [ ]:
from pathlib import Path
import os
import pandas as pd

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

def find_data_dir() -> Path:
    """Locate the competition data on Kaggle, or fall back to a local folder."""
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        hits = sorted(kaggle_input.rglob("train_series.csv"))
        if hits:
            return hits[0].parent
    return Path(os.environ.get("DATA_DIR", "../data")).resolve()

DATA_DIR = find_data_dir()
ON_KAGGLE = str(DATA_DIR).startswith("/kaggle")
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs/01_eda")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data:", DATA_DIR, "| on Kaggle:", ON_KAGGLE)
print(sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
series = pd.read_csv(DATA_DIR / "train_series.csv")
print(train.shape, series.shape)
train.head()

## Labels: how many studies are labelled, and how common is each finding?

In [ ]:
labels = [c for c in LABELS if c in train.columns]
labelled = train[labels].notna().all(axis=1)
print(f"Labelled studies: {labelled.sum()} / {len(train)}")
prevalence = train.loc[labelled, labels].mean().sort_values(ascending=False)
prevalence.to_csv(OUT_DIR / "label_prevalence.csv")
prevalence

## Series per study and plane / sequence mix

In [ ]:
print(series.groupby("StudyInstanceUID").size().describe())
series.groupby(["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]).size()

## Example reports

In [ ]:
for r in train["Report"].dropna().sample(3, random_state=0):
    print(r[:800], "\n" + "-" * 80)